In [ ]:
import torch
import torchaudio
import os
import gc
import glob
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import WavLMModel, HubertModel, Wav2Vec2Model

class SpeakerDataset(Dataset):
    def __init__(self, folder_path):
        self.file_paths = glob.glob(os.path.join(folder_path, "**", "*.wav"), recursive=True)
        self.file_paths.sort(key=lambda x: os.path.getsize(x))
        
    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        waveform, _ = torchaudio.load(path)
        
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0)
        else:
            waveform = waveform.squeeze(0)
            
        mean = waveform.mean()
        var = waveform.var(unbiased=False)
        waveform = (waveform - mean) / torch.sqrt(var + 1e-7)

        filename = os.path.basename(path)
        speaker_id = filename.split('_')[0]
        return waveform, speaker_id, filename

def collate_fn(batch):
    waveforms, ids, names = zip(*batch)
    padded_waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True, padding_value=0.0)
    lengths = torch.tensor([len(w) for w in waveforms])
    max_len = padded_waveforms.shape[1]
    attention_mask = torch.arange(max_len).expand(len(lengths), max_len) < lengths.unsqueeze(1)
    return padded_waveforms, attention_mask.long(), list(ids), list(names)

@torch.inference_mode()
def run_extraction_robust(model_key, folder_path, save_dir, batch_size=16, checkpoint_size=10000):
    os.makedirs(save_dir, exist_ok=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    final_save_path = os.path.join(save_dir, f"{model_key}_full.pt")
    if os.path.exists(final_save_path):
        print(f"⏩ File đích {final_save_path} đã tồn tại. Bỏ qua hoàn toàn model {model_key}.")
        return

    dataset = SpeakerDataset(folder_path)
    total_files = len(dataset)
    num_checkpoints = (total_files + checkpoint_size - 1) // checkpoint_size

    model_map = {"wavlm": WavLMModel, "hubert": HubertModel, "wav2vec2": Wav2Vec2Model}
    repo_map = {"wavlm": "microsoft/wavlm-base", "hubert": "facebook/hubert-base-ls960", "wav2vec2": "facebook/wav2vec2-base-960h"}
    model_class, repo = model_map[model_key], repo_map[model_key]
    model = None

    print(f"--- Bắt đầu trích xuất an toàn cho {model_key.upper()} ---")

    for ckpt_idx in range(num_checkpoints):
        ckpt_path = os.path.join(save_dir, f"{model_key}_temp_ckpt_{ckpt_idx}.pt")
        
        if os.path.exists(ckpt_path):
            print(f"⏩ Checkpoint {ckpt_idx}/{num_checkpoints-1} đã hoàn thành từ trước. Resume (Chạy tiếp)...")
            continue

        if model is None:
            # FIX: Đã gỡ bỏ tham số attn_implementation="sdpa" gây lỗi
            model = model_class.from_pretrained(
                repo, 
                output_hidden_states=True, 
                torch_dtype=torch.bfloat16
            ).to(device).eval()

        print(f"🚀 Đang xử lý Checkpoint {ckpt_idx}/{num_checkpoints-1}...")
        start_idx = ckpt_idx * checkpoint_size
        end_idx = min(start_idx + checkpoint_size, total_files)
        subset = Subset(dataset, list(range(start_idx, end_idx)))
        
        dataloader = DataLoader(subset, batch_size=batch_size, collate_fn=collate_fn, num_workers=0, pin_memory=True)
        ckpt_embeddings, ckpt_ids, ckpt_names = [], [], []

        for waveforms, attention_mask, ids, names in tqdm(dataloader, desc=f"CKPT {ckpt_idx}"):
            waveforms = waveforms.to(device, dtype=torch.bfloat16, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            
            try:
                outputs = model(input_values=waveforms, attention_mask=attention_mask, output_hidden_states=True)
                stacked = torch.stack(outputs.hidden_states)
                pooled = stacked.mean(dim=2).permute(1, 0, 2).cpu().float()
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                temp_pooled = []
                for i in range(len(waveforms)):
                    w = waveforms[i].unsqueeze(0)
                    mask = attention_mask[i].unsqueeze(0)
                    out = model(input_values=w, attention_mask=mask, output_hidden_states=True)
                    p = torch.stack(out.hidden_states).mean(dim=2).permute(1, 0, 2).cpu().float()
                    temp_pooled.append(p)
                    torch.cuda.empty_cache()
                pooled = torch.cat(temp_pooled, dim=0)

            ckpt_embeddings.append(pooled)
            ckpt_ids.extend(ids)
            ckpt_names.extend(names)

        torch.save({
            'embeddings': torch.cat(ckpt_embeddings, dim=0),
            'speaker_ids': ckpt_ids,
            'filenames': ckpt_names
        }, ckpt_path)
        
        del ckpt_embeddings, ckpt_ids, ckpt_names
        gc.collect()
        torch.cuda.empty_cache()

    if model is not None:
        del model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"🔄 Đang gom các checkpoint thành file duy nhất: {final_save_path}")
    all_embeddings, all_ids, all_names = [], [], []
    
    for ckpt_idx in range(num_checkpoints):
        ckpt_path = os.path.join(save_dir, f"{model_key}_temp_ckpt_{ckpt_idx}.pt")
        data = torch.load(ckpt_path, weights_only=False)
        all_embeddings.append(data['embeddings'])
        all_ids.extend(data['speaker_ids'])
        all_names.extend(data['filenames'])
        
    torch.save({
        'embeddings': torch.cat(all_embeddings, dim=0),
        'speaker_ids': all_ids,
        'filenames': all_names,
        'model_name': model_key
    }, final_save_path)

    for ckpt_idx in range(num_checkpoints):
        ckpt_path = os.path.join(save_dir, f"{model_key}_temp_ckpt_{ckpt_idx}.pt")
        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)

    print(f"✅ Đã dọn dẹp file tạm và hoàn tất lưu {model_key}_full.pt!")


if __name__ == "__main__":
    AUDIO_FOLDER = r"D:\Study\7-SP26\DATxSLP\Test set O\test-O"  # Thay bằng thư mục của bạn
    OUTPUT_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\test"   # Thư mục lưu kết quả

    models_to_run = ["wavlm", "hubert", "wav2vec2"]
    
    print(f"🚀 BẮT ĐẦU CHẠY PIPELINE CHO {len(models_to_run)} MODELS: {models_to_run}\n")

    for current_model in models_to_run:
        print(f"{'='*50}")
        print(f"▶ ĐANG XỬ LÝ MODEL: {current_model.upper()}")
        print(f"{'='*50}")
        
        run_extraction_robust(
            model_key=current_model, 
            folder_path=AUDIO_FOLDER, 
            save_dir=OUTPUT_DIR, 
            batch_size=16,
            checkpoint_size=10000
        )
        print("\n")

    print("🎉 ĐÃ HOÀN THÀNH TOÀN BỘ PIPELINE! Các file _full.pt đã sẵn sàng.")